In [1]:
%run sagemaker_utils.ipynb

  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [62 lines of output]
      /tmp/pip-build-env-z19vwh5f/overlay/lib/python3.12/site-packages/setuptools/dist.py:765: SetuptoolsDeprecationWarning: License classifiers are deprecated.
      !!
      
              ********************************************************************************
              Please consider removing the following classifiers in favor of a SPDX license expression:
      
              License :: OSI Approved :: Apache Software License
      
              See https://packaging.python.org/en/latest/guides/writing-pyproject-toml/#license for details.
              ********************************************************************************
      
      !!
        self._finalize_license_expression()
      running bdist_wheel
      running build
      running build_py
      creating build/lib.linux-x86_64-cpython-312/

In [2]:
import json

# Bert TensorRT

In [77]:
!tar -C triton-serve-trt-g5/ -czf model.tar.gz bert
model_uri = sagemaker_session.upload_data(path="model.tar.gz", key_prefix="triton-serve-pt")

In [78]:
sm_model_name = "triton-nlp-bert-trt-benchmark"
endpoint_config_name = "triton-nlp-bert-trt-benchmark"
endpoint_name = "triton-nlp-bert-trt-benchmark-2"

In [ ]:
create_model(sm_model_name, model_uri, "bert")
create_endpoint_config(endpoint_config_name, sm_model_name, instance_type="ml.g5.4xlarge")
create_endpoint(endpoint_name, endpoint_config_name)
poll(endpoint_name)

Model Arn: arn:aws:sagemaker:ap-south-1:978983596161:model/triton-nlp-bert-trt-benchmark
Endpoint Config Arn: arn:aws:sagemaker:ap-south-1:978983596161:endpoint-config/triton-nlp-bert-trt-benchmark
Endpoint Arn: arn:aws:sagemaker:ap-south-1:978983596161:endpoint/triton-nlp-bert-trt-benchmark-2
Status: Creating
Status: Creating


In [76]:
cleanup(sm_model_name, endpoint_config_name, endpoint_name)

# Bert Tokenizer (no model)

In [31]:
!tar -C triton-serve-tokenizer/ -czf model.tar.gz tokenizer
model_uri = sagemaker_session.upload_data(path="model2.tar.gz", key_prefix="triton-serve-pt")

In [53]:
sm_model_name = "triton-nlp-tokenizer-benchmark"
endpoint_config_name = "triton-nlp-tokenizer-benchmark"
endpoint_name = "triton-nlp-tokenizer-benchmark-2"

In [33]:
create_model(sm_model_name, model_uri, "tokenizer")
create_endpoint_config(endpoint_config_name, sm_model_name)
create_endpoint(endpoint_name, endpoint_config_name)
poll(endpoint_name)

Model Arn: arn:aws:sagemaker:ap-south-1:978983596161:model/triton-nlp-tokenizer-benchmark
Endpoint Config Arn: arn:aws:sagemaker:ap-south-1:978983596161:endpoint-config/triton-nlp-tokenizer-benchmark
Endpoint Arn: arn:aws:sagemaker:ap-south-1:978983596161:endpoint/triton-nlp-tokenizer-benchmark-2
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: InService
Arn: arn:aws:sagemaker:ap-south-1:978983596161:endpoint/triton-nlp-tokenizer-benchmark-2
Status: InService


In [10]:
sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
sm.delete_endpoint(EndpointName=endpoint_name)

{'ResponseMetadata': {'RequestId': '522e593a-d5e9-468c-980f-db5e81bfa9fa',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': '522e593a-d5e9-468c-980f-db5e81bfa9fa',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'date': 'Sun, 22 Mar 2026 17:19:27 GMT',
   'content-length': '0'},
  'RetryAttempts': 0}}

In [ ]:
text_triton = "Triton Inference Server provides a cloud and edge inferencing solution optimized for both CPUs and GPUs."

payload = {
    "inputs": [
        {
            "name": "text",              # must match config.pbtxt
            "shape": [1, 1],
            "datatype": "BYTES",
            "data": [[text_triton]]
        }
    ]
}

response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   # important change
    Body=json.dumps(payload)
)

result = json.loads(response["Body"].read().decode("utf8"))
print(result)

In [54]:
cleanup(sm_model_name, endpoint_config_name, endpoint_name)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 cleanup(sm_model_name, endpoint_config_name, endpoint_name)                                  │
│   2                                                                                              │
│                                                                                                  │
│ in cleanup:4                                                                                     │
│                                                                                                  │
│   1 def cleanup(sm_model_name, endpoint_config_name, endpoint_name):                             │
│   2 │   sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)                       │
│   3 │   sm.delete_model(ModelName=sm_model_name)                                                 │
│ ❱ 4 │   sm.delete_endpoint(EndpointName=endpoint_name)                                           │
│   5                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:569 in _api_call                      │
│                                                                                                  │
│    566 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    567 │   │   │   │   )                                                                         │
│    568 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  569 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    570 │   │                                                                                     │
│    571 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    572                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:1023 in _make_api_call                │
│                                                                                                  │
│   1020 │   │   │   │   "Code"                                                                    │
│   1021 │   │   │   )                                                                             │
│   1022 │   │   │   error_class = self.exceptions.from_code(error_code)                           │
│ ❱ 1023 │   │   │   raise error_class(parsed_response, operation_name)                            │
│   1024 │   │   else:                                                                             │
│   1025 │   │   │   return parsed_response                                                        │
│   1026                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ClientError: An error occurred (ValidationException) when calling the DeleteEndpoint operation: Could not find 
endpoint "triton-nlp-tokenizer-benchmark-2".

# bert TensorRT with tokenizer

In [72]:
!mkdir -p triton-serve-tokenizer-trt-g5/bert_ensemble/1/
!tar -C triton-serve-tokenizer-trt/ -czf model.tar.gz bert bert_tokenizer bert_ensemble
model_uri = sagemaker_session.upload_data(path="model3.tar.gz", key_prefix="triton-serve-pt")

^C


In [73]:
sm_model_name = "triton-nlp-bert-with-tokenizer-benchmark"
endpoint_config_name = "triton-nlp-bert-with-tokenizer-benchmark"
endpoint_name = "triton-nlp-bert-with-tokenizer-benchmark-2"

In [74]:
create_model(sm_model_name, model_uri, "bert_ensemble")
create_endpoint_config(endpoint_config_name, sm_model_name, instance_type="ml.g5.4xlarge")
create_endpoint(endpoint_name, endpoint_config_name)

Model Arn: arn:aws:sagemaker:ap-south-1:978983596161:model/triton-nlp-bert-with-tokenizer-benchmark
Endpoint Config Arn: arn:aws:sagemaker:ap-south-1:978983596161:endpoint-config/triton-nlp-bert-with-tokenizer-benchmark


╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:3                                                                                    │
│                                                                                                  │
│   1 create_model(sm_model_name, model_uri, "bert_ensemble")                                      │
│   2 create_endpoint_config(endpoint_config_name, sm_model_name)                                  │
│ ❱ 3 create_endpoint(endpoint_name, endpoint_config_name)                                         │
│   4                                                                                              │
│                                                                                                  │
│ in create_endpoint:2                                                                             │
│                                                                                                  │
│   1 def create_endpoint(endpoint_name, endpoint_config_name):                                    │
│ ❱ 2 │   create_endpoint_response = sm.create_endpoint(                                           │
│   3 │   │   EndpointName=endpoint_name, EndpointConfigName=endpoint_config_name                  │
│   4 │   )                                                                                        │
│   5                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:569 in _api_call                      │
│                                                                                                  │
│    566 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    567 │   │   │   │   )                                                                         │
│    568 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  569 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    570 │   │                                                                                     │
│    571 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    572                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:1005 in _make_api_call                │
│                                                                                                  │
│   1002 │   │   │   │   self.meta.config, request_dict, operation_model                           │
│   1003 │   │   │   )                                                                             │
│   1004 │   │   │   apply_request_checksum(request_dict)                                          │
│ ❱ 1005 │   │   │   http, parsed_response = self._make_request(                                   │
│   1006 │   │   │   │   operation_model, request_dict, request_context                            │
│   1007 │   │   │   )                                                                             │
│   1008                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:1029 in _make_request                 │
│                                                                                                  │
│   1026 │                                                                                         │
│   1027 │   def _make_request(self, operation_model, request

In [ ]:
poll(endpoint_name)

In [9]:
text_triton = "Triton Inference Server provides a cloud and edge inferencing solution optimized for both CPUs and GPUs."

payload = {
    "inputs": [
        {
            "name": "text",              # must match config.pbtxt
            "shape": [8, 1],
            "datatype": "BYTES",
            "data": [[text_triton] * 8]
        }
    ]
}

response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   # important change
    Body=json.dumps(payload)
)

result = json.loads(response["Body"].read().decode("utf8"))
print(result)

{'model_name': 'tokenizer', 'model_version': '1', 'outputs': [{'name': 'token_ids', 'datatype': 'INT32', 'shape': [8, 128], 'data': [101, 13012, 2669, 28937, 8241, 3640, 1037, 6112, 1998, 3341, 1999, 7512, 2368, 6129, 5576, 23569, 27605, 5422, 2005, 2119, 17368, 2015, 1998, 14246, 2271, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 101, 13012, 2669, 28937, 8241, 3640, 1037, 6112, 1998, 3341, 1999, 7512, 2368, 6129, 5576, 23569, 27605, 5422, 2005, 2119, 17368, 2015, 1998, 14246, 2271, 1012, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [56]:
cleanup(sm_model_name, endpoint_config_name, endpoint_name)

╭─────────────────────────────── Traceback (most recent call last) ────────────────────────────────╮
│ in <module>:1                                                                                    │
│                                                                                                  │
│ ❱ 1 cleanup(sm_model_name, endpoint_config_name, endpoint_name)                                  │
│   2                                                                                              │
│                                                                                                  │
│ in cleanup:2                                                                                     │
│                                                                                                  │
│   1 def cleanup(sm_model_name, endpoint_config_name, endpoint_name):                             │
│ ❱ 2 │   sm.delete_endpoint_config(EndpointConfigName=endpoint_config_name)                       │
│   3 │   sm.delete_model(ModelName=sm_model_name)                                                 │
│   4 │   sm.delete_endpoint(EndpointName=endpoint_name)                                           │
│   5                                                                                              │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:569 in _api_call                      │
│                                                                                                  │
│    566 │   │   │   │   │   f"{py_operation_name}() only accepts keyword arguments."              │
│    567 │   │   │   │   )                                                                         │
│    568 │   │   │   # The "self" in this scope is referring to the BaseClient.                    │
│ ❱  569 │   │   │   return self._make_api_call(operation_name, kwargs)                            │
│    570 │   │                                                                                     │
│    571 │   │   _api_call.__name__ = str(py_operation_name)                                       │
│    572                                                                                           │
│                                                                                                  │
│ /opt/conda/lib/python3.12/site-packages/botocore/client.py:1023 in _make_api_call                │
│                                                                                                  │
│   1020 │   │   │   │   "Code"                                                                    │
│   1021 │   │   │   )                                                                             │
│   1022 │   │   │   error_class = self.exceptions.from_code(error_code)                           │
│ ❱ 1023 │   │   │   raise error_class(parsed_response, operation_name)                            │
│   1024 │   │   else:                                                                             │
│   1025 │   │   │   return parsed_response                                                        │
│   1026                                                                                           │
╰──────────────────────────────────────────────────────────────────────────────────────────────────╯
ClientError: An error occurred (ValidationException) when calling the DeleteEndpointConfig operation: Could not 
find endpoint configuration "triton-nlp-bert-with-tokenizer-benchmark".

# Bert with IM requirement

In [67]:
!tar -C triton-serve-similarity/ -czf model.tar.gz bert bert_tokenizer cosine_similarity similarity_ensemble
model_uri = sagemaker_session.upload_data(path="model.tar.gz", key_prefix="triton-serve-pt")

In [68]:
sm_model_name = "triton-nlp-bert-with-imreq-benchmark"
endpoint_config_name = "triton-nlp-bert-with-imreq-benchmark"
endpoint_name = "triton-nlp-bert-with-imreq-benchmark"

In [69]:
create_model(sm_model_name, model_uri, "similarity_ensemble")
create_endpoint_config(endpoint_config_name, sm_model_name)
create_endpoint(endpoint_name, endpoint_config_name)
poll(endpoint_name)

Model Arn: arn:aws:sagemaker:ap-south-1:978983596161:model/triton-nlp-bert-with-imreq-benchmark
Endpoint Config Arn: arn:aws:sagemaker:ap-south-1:978983596161:endpoint-config/triton-nlp-bert-with-imreq-benchmark
Endpoint Arn: arn:aws:sagemaker:ap-south-1:978983596161:endpoint/triton-nlp-bert-with-imreq-benchmark
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: Creating
Status: InService
Arn: arn:aws:sagemaker:ap-south-1:978983596161:endpoint/triton-nlp-bert-with-imreq-benchmark
Status: InService


In [71]:
query = "what is ai?"
answers = ["next token prediction", "it's not ai", "what is ai?", "openai google anthropic"]

payload = {
    "inputs": [
        {
            "name": "text",              # must match config.pbtxt
            "shape": [5, 1],
            "datatype": "BYTES",
            "data": [[query] + answers]
        }
    ]
}

response = client.invoke_endpoint(
    EndpointName=endpoint_name,
    ContentType="application/json",   # important change
    Body=json.dumps(payload)
)

result = json.loads(response["Body"].read().decode("utf8"))
print(result)

{'model_name': 'similarity_ensemble', 'model_version': '1', 'parameters': {'sequence_id': 0, 'sequence_start': False, 'sequence_end': False}, 'outputs': [{'name': 'scores', 'datatype': 'FP32', 'shape': [4], 'data': [0.8262747526168823, 0.9191592335700989, 0.9999999403953552, 0.8648236989974976]}]}


In [ ]:
cleanup(sm_model_name, endpoint_config_name, endpoint_name)